In [1]:
import pandas as pd

# Setup & Data Loading

In [2]:
df = pd.read_csv("dataset_sig_phq9.csv")

In [3]:
# Fix separation-causing sparse categories before modeling.
# PHQ_class (the outcome) is never touched -- all 5 categories are kept as-is.
# Only sparse levels of predictor variables that produce a zero cell against
# PHQ_class (causing perfect separation / non-convergence) are merged into an
# adjacent level of the same predictor:
# - D3_device "Tablet" has 0 respondents in "Severe depressive" -> merge into "Computer/Laptop"
# - G6_panic_attack "Always" has 0 respondents in "Mild depressive" (the planned reference
#   class), which would cause perfect separation in every equation -> merge into "Often"
# - F1_academic_satisfaction "Highly satisfactory" has 0 respondents in "Severe depressive"
#   -> merge into adjacent "Moderately satisfactory"
# - G4_isolation "Always" has 0 respondents in "Minimal depressive" and in
#   "Moderately Severe depressive" -> merge into adjacent "Often"
# - G5_negative_mental "Not sure" has 0 respondents in "Severe depressive"
#   -> merge into adjacent "Occasionally"
# - G2_anxious_without_device "Always" has 0 respondents in "Moderately Severe depressive"
#   -> merge into adjacent "Often"
df['D3_device'] = df['D3_device'].replace({'Tablet': 'Computer/Laptop'})
df['G6_panic_attack'] = df['G6_panic_attack'].replace({'Always': 'Often'})
df['F1_academic_satisfaction'] = df['F1_academic_satisfaction'].replace(
    {'Highly satisfactory': 'Moderately satisfactory'}
)
df['G4_isolation'] = df['G4_isolation'].replace({'Always': 'Often'})
df['G5_negative_mental'] = df['G5_negative_mental'].replace({'Not sure': 'Occasionally'})
df['G2_anxious_without_device'] = df['G2_anxious_without_device'].replace({'Always': 'Often'})

print("PHQ_class categories (unchanged):", df['PHQ_class'].unique().tolist())

PHQ_class categories (unchanged): ['Moderate depressive', 'Moderately Severe depressive', 'Severe depressive', 'Mild depressive', 'Minimal depressive']


In [4]:
df.columns

Index(['PHQ_class', 'age_class', 'B6_relationship_with_family', 'weight_class',
       'D3_device', 'D5_parents_screen_time', 'F1_academic_satisfaction',
       'G1_mood_swings', 'G2_anxious_without_device', 'G4_isolation',
       'G5_negative_mental', 'G6_panic_attack', 'H1_mobile_while_eating'],
      dtype='str')

In [5]:
# Check target variable distribution
print("Target variable: PHQ_class")
print(df['PHQ_class'].value_counts())
print("\nShape:", df.shape)
print("\nMissing values:\n", df.isnull().sum().sum())

Target variable: PHQ_class
PHQ_class
Mild depressive                 163
Moderate depressive             142
Minimal depressive               86
Moderately Severe depressive     46
Severe depressive                15
Name: count, dtype: int64

Shape: (452, 13)

Missing values:
 0


# Multinomial Logistic Regression
**Target variable:** `PHQ_class`

---

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [7]:
#  1. Prepare features and target
target = 'PHQ_class'
features = [c for c in df.columns if c != target]

X = df[features].copy()
y = df[target].copy()

# Encode every categorical column with a fresh LabelEncoder
encoders = {}
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

le_target = LabelEncoder()
y_enc = le_target.fit_transform(y.astype(str))
class_names = le_target.classes_

print("Classes:", class_names)
print("Encoded labels:", np.unique(y_enc))

Classes: ['Mild depressive' 'Minimal depressive' 'Moderate depressive'
 'Moderately Severe depressive' 'Severe depressive']
Encoded labels: [0 1 2 3 4]


In [8]:

# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

Train size: 361  |  Test size: 91


In [9]:

# 3. Fit Multinomial Logistic Regression
mlr = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
mlr.fit(X_train, y_train)
y_pred = mlr.predict(X_test)

# 4. Evaluation 
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

Test Accuracy: 0.3626 (36.26%)

Classification Report:
                              precision    recall  f1-score   support

             Mild depressive       0.38      0.45      0.42        33
          Minimal depressive       0.37      0.41      0.39        17
         Moderate depressive       0.35      0.38      0.37        29
Moderately Severe depressive       0.00      0.00      0.00         9
           Severe depressive       0.00      0.00      0.00         3

                    accuracy                           0.36        91
                   macro avg       0.22      0.25      0.23        91
                weighted avg       0.32      0.36      0.34        91



## Odds Ratios
**Per class vs. reference class (`Mild depressive`)**

In [10]:

# Odds Ratios 
# OR = exp(coefficient); OR > 1 → increases odds of that class, < 1 → decreases
odds_ratio_df = pd.DataFrame(
    np.exp(mlr.coef_),
    index=class_names,
    columns=features
).T

print("Odds Ratios per class (rows=features, columns=PHQ_class classes):\n")
print(odds_ratio_df.round(4).to_string())

Odds Ratios per class (rows=features, columns=PHQ_class classes):

                             Mild depressive  Minimal depressive  Moderate depressive  Moderately Severe depressive  Severe depressive
age_class                             0.7730              2.0026               0.8662                        0.8140             0.9163
B6_relationship_with_family           0.5524              0.7374               0.9139                        0.9064             2.9638
weight_class                          0.9800              0.8393               0.7820                        1.2926             1.2028
D3_device                             1.1144              0.9690               1.0044                        0.7829             1.1778
D5_parents_screen_time                1.0883              0.5310               1.5429                        1.0180             1.1019
F1_academic_satisfaction              1.2019              1.0433               1.1246                        0.9580        

In [11]:
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit

# Refit using statsmodels to get standard errors, p-values, and CIs
X_sm = sm.add_constant(X.values.astype(float))
feature_names_sm = ['const'] + features

mnl = MNLogit(y_enc, X_sm)
result = mnl.fit(method='bfgs', maxiter=2000, disp=False)

ref_class = class_names[0]
eq_classes = class_names[1:]

rows = []
for eq_idx, cls in enumerate(eq_classes):
    coefs = result.params[:, eq_idx]
    se    = result.bse[:, eq_idx]
    pvals = result.pvalues[:, eq_idx]

    for feat_idx, feat in enumerate(feature_names_sm):
        if feat == 'const':
            continue
        b = coefs[feat_idx]
        s = se[feat_idx]
        rows.append({
            'Class (vs. reference)': f"{cls}  vs.  {ref_class}",
            'Feature':               feat,
            'Odds Ratio':            round(np.exp(b), 4),
            'Lower 95% CI':          round(np.exp(b - 1.96 * s), 4),
            'Upper 95% CI':          round(np.exp(b + 1.96 * s), 4),
            'p-value':               round(pvals[feat_idx], 4),
        })

or_table = pd.DataFrame(rows)

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.4f}'.format)
print("Odds Ratio Table — Multinomial Logistic Regression")
print(f"Reference class: '{ref_class}'\n")
print(or_table.to_string(index=False))

Odds Ratio Table — Multinomial Logistic Regression
Reference class: 'Mild depressive'

                             Class (vs. reference)                     Feature  Odds Ratio  Lower 95% CI  Upper 95% CI  p-value
          Minimal depressive  vs.  Mild depressive                   age_class      2.3454        1.4913        3.6888   0.0002
          Minimal depressive  vs.  Mild depressive B6_relationship_with_family      1.3374        0.5909        3.0273   0.4855
          Minimal depressive  vs.  Mild depressive                weight_class      0.8746        0.5995        1.2760   0.4869
          Minimal depressive  vs.  Mild depressive                   D3_device      0.9022        0.4576        1.7788   0.7664
          Minimal depressive  vs.  Mild depressive      D5_parents_screen_time      0.5249        0.2732        1.0084   0.0530
          Minimal depressive  vs.  Mild depressive    F1_academic_satisfaction      0.8196        0.5919        1.1349   0.2309
          Minimal

In [12]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

output_pdf = "report_v3/odds_ratio_table_phq9.pdf"

col_widths = [0.30, 0.22, 0.12, 0.13, 0.13, 0.10]
row_height = 0.32   # inches per row
header_height = 0.55
rows_per_page = 40

pages = [or_table.iloc[i:i+rows_per_page] for i in range(0, len(or_table), rows_per_page)]

with PdfPages(output_pdf) as pdf:
    for page_num, page_df in enumerate(pages):
        n_rows = len(page_df)
        fig_h = header_height + n_rows * row_height + 1.2
        fig, ax = plt.subplots(figsize=(14, fig_h))
        ax.axis('off')

        table_data = [or_table.columns.tolist()] + page_df.values.tolist()
        col_labels = or_table.columns.tolist()

        tbl = ax.table(
            cellText=page_df.values.tolist(),
            colLabels=col_labels,
            colWidths=col_widths,
            loc='center',
            cellLoc='center',
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7.5)
        tbl.scale(1, 1.4)

        # Style header
        for col_idx in range(len(col_labels)):
            cell = tbl[0, col_idx]
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')

        # Alternating row shading
        for row_idx in range(1, n_rows + 1):
            for col_idx in range(len(col_labels)):
                cell = tbl[row_idx, col_idx]
                cell.set_facecolor('#f2f2f2' if row_idx % 2 == 0 else 'white')
                cell.set_edgecolor('#cccccc')

        total_pages = len(pages)
        ax.set_title(
            f"Odds Ratio Table — Multinomial Logistic Regression\n"
            f"Reference class: '{ref_class}'   |   Page {page_num+1} of {total_pages}",
            fontsize=10, fontweight='bold', pad=12
        )
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f"Saved: {output_pdf}  ({len(pages)} page(s), {len(or_table)} rows)")

Saved: report_v3/odds_ratio_table_phq9.pdf  (2 page(s), 48 rows)


## Goodness of Fit
**Multinomial Logistic Regression — model diagnostics**

In [13]:
# Likelihood Ratio Test & Pseudo R²

llr      = result.llr
llr_pval = result.llr_pvalue
mcfadden = result.prsquared          # McFadden's pseudo-R²

print("=" * 55)
print("  Goodness of Fit — Multinomial Logistic Regression")
print("=" * 55)
print(f"\n1. Likelihood Ratio Test")
print(f"   LR statistic : {llr:.4f}")
print(f"   p-value      : {llr_pval:.4e}")
print(f"   Interpretation: {'Model fits significantly better than null (intercept-only)' if llr_pval < 0.05 else 'No significant improvement over null model'}")

print(f"\n2. Pseudo R² (McFadden)")
print(f"   McFadden R²  : {mcfadden:.4f}")
if mcfadden < 0.1:
    interpretation = "Poor fit"
elif mcfadden < 0.2:
    interpretation = "Acceptable fit"
elif mcfadden < 0.4:
    interpretation = "Good fit  (0.2–0.4 is typical for MLR)"
else:
    interpretation = "Very good fit"
print(f"   Interpretation: {interpretation}")
print("=" * 55)

  Goodness of Fit — Multinomial Logistic Regression

1. Likelihood Ratio Test
   LR statistic : 137.0633
   p-value      : 1.6717e-10
   Interpretation: Model fits significantly better than null (intercept-only)

2. Pseudo R² (McFadden)
   McFadden R²  : 0.1089
   Interpretation: Acceptable fit


In [14]:

# Goodness of Fit — Observed vs. Expected Counts
from scipy.stats import chi2

y_pred_proba = mlr.predict_proba(X)

observed = pd.Series(le_target.inverse_transform(y_enc)).value_counts().reindex(class_names)
expected = pd.Series(y_pred_proba.sum(axis=0), index=class_names)

crosstab_gof = pd.DataFrame({
    'Observed (O)': observed.values.astype(int),
    'Expected (E)': expected.round(2).values,
}, index=class_names)
crosstab_gof.index.name = 'PHQ_class'

print(crosstab_gof.to_string())

chi2_stat = (((observed.values - expected.values) ** 2) / expected.values).sum()
df_chi    = len(class_names) - 1
p_chi     = 1 - chi2.cdf(chi2_stat, df_chi)

print(f"\nChi-square statistic : {chi2_stat:.4f}")
print(f"Degrees of freedom   : {df_chi}")
print(f"p-value              : {p_chi:.4f}")
print(f"Interpretation       : {'Model fits well (no significant discrepancy)' if p_chi >= 0.05 else 'Significant discrepancy between observed and expected'}")

                              Observed (O)  Expected (E)
PHQ_class                                               
Mild depressive                        163      162.7900
Minimal depressive                      86       86.7900
Moderate depressive                    142      141.2800
Moderately Severe depressive            46       46.0000
Severe depressive                       15       15.1300

Chi-square statistic : 0.0124
Degrees of freedom   : 4
p-value              : 1.0000
Interpretation       : Model fits well (no significant discrepancy)


# Publication-Style Results Table (Indicator Coding)
**Journal-format table — categorical predictors with reference levels, one column group per outcome class (vs. `Mild depressive`)**

In [ ]:
# Reference (baseline) level chosen for every categorical predictor
ref_levels = {
    'age_class':                   '11-13 years',
    'B6_relationship_with_family': 'Good',
    'weight_class':                'Medium (45-60 kg)',
    'D3_device':                   'Mobile phone',
    'D5_parents_screen_time':      'Less than 3 hours',
    'F1_academic_satisfaction':    'Satisfactory',
    'G1_mood_swings':              'Not at all',
    'G2_anxious_without_device':   'Not at all',
    'G4_isolation':                'Never',
    'G5_negative_mental':          'Never',
    'G6_panic_attack':             'Never',
    'H1_mobile_while_eating':      'Never',
}

# Build indicator-coded design matrix, one indicator column per non-reference level
X_cat = df[features].copy()
indicator_frames = []
var_group_map = {}   # feature -> list of (indicator_col, level_label)

for feat in features:
    ref = ref_levels[feat]
    levels = [l for l in X_cat[feat].unique().tolist() if l != ref]
    levels.sort()
    indicators = pd.get_dummies(X_cat[feat], prefix=feat, drop_first=False)
    keep_cols = [f"{feat}_{lvl}" for lvl in levels]
    indicators = indicators[keep_cols].astype(float)
    indicator_frames.append(indicators)
    var_group_map[feat] = list(zip(keep_cols, levels))

X_indicators = pd.concat(indicator_frames, axis=1)
print("Indicator-coded design matrix shape:", X_indicators.shape)
X_indicators.head()


In [ ]:
# Fit the indicator-coded model with Firth's bias-reduced (penalized) logistic
# regression, one binary equation per outcome class vs. the reference class.
#
# Why: plain MLE (MNLogit) on the full indicator matrix produces astronomically
# large odds ratios and confidence intervals for several rows -- e.g. some
# "Severe depressive" (n=15) coefficients came out with OR in the thousands
# and CI upper bounds in the billions. This is quasi-complete separation:
# with only 15 respondents in that class spread across 43 indicator predictor
# columns, many cells have 0-3 observations, so the unpenalized likelihood
# surface is nearly flat and the optimizer can push coefficients arbitrarily
# far without changing the fit. Firth's method adds a Jeffreys-prior bias
# correction to the score equations, which is the standard statistical fix
# for rare-outcome / sparse-data separation and shrinks these coefficients to
# realistic values without merging any PHQ_class categories or predictor levels.
import numpy as np
from scipy.stats import norm


def _penalized_loglik(X, y, beta):
    eta = np.clip(X @ beta, -30, 30)
    pr = 1 / (1 + np.exp(-eta))
    W = np.clip(pr * (1 - pr), 1e-10, None)
    XtWX = X.T * W @ X
    sign, logdet = np.linalg.slogdet(XtWX)
    ll = np.sum(y * np.log(np.clip(pr, 1e-12, 1)) + (1 - y) * np.log(np.clip(1 - pr, 1e-12, 1)))
    return ll + 0.5 * logdet if sign > 0 else -np.inf


def firth_logit(X, y, max_iter=500, tol=1e-6):
    """Firth bias-reduced logistic regression via penalized IRLS with step-halving."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    beta = np.zeros(X.shape[1])
    prev_pll = _penalized_loglik(X, y, beta)

    for _ in range(max_iter):
        eta = np.clip(X @ beta, -30, 30)
        pr = 1 / (1 + np.exp(-eta))
        W = np.clip(pr * (1 - pr), 1e-10, None)
        XtWX = X.T * W @ X
        try:
            XtWX_inv = np.linalg.inv(XtWX)
        except np.linalg.LinAlgError:
            XtWX_inv = np.linalg.pinv(XtWX)

        WX = X * W[:, None]
        H = np.einsum('ij,jk,ik->i', WX, XtWX_inv, X)  # hat diagonal
        U = X.T @ (y - pr + H * (0.5 - pr))             # Firth-adjusted score
        delta = XtWX_inv @ U

        step = 1.0
        for _ in range(30):
            beta_new = beta + step * delta
            pll_new = _penalized_loglik(X, y, beta_new)
            if np.isfinite(pll_new) and pll_new >= prev_pll - 1e-10:
                break
            step /= 2
        else:
            beta_new = beta

        max_change = np.max(np.abs(beta_new - beta))
        ll_change = abs(pll_new - prev_pll)
        beta = beta_new
        prev_pll = pll_new
        if max_change < tol or ll_change < 1e-8:
            break

    eta = np.clip(X @ beta, -30, 30)
    pr = 1 / (1 + np.exp(-eta))
    W = np.clip(pr * (1 - pr), 1e-10, None)
    XtWX = X.T * W @ X
    try:
        cov = np.linalg.inv(XtWX)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(XtWX)
    return beta, cov


X_design = np.column_stack([np.ones(len(df)), X_indicators.values.astype(float)])
indicator_col_names = ['const'] + X_indicators.columns.tolist()

eq_classes = class_names[1:]   # classes vs. reference 'Mild depressive'

# Collect OR / CI / p-value for every indicator, per outcome class
indicator_stats = {}   # indicator_col -> {cls: (OR, lo, hi, p)}
convergence_log = []
for cls in eq_classes:
    mask = (df['PHQ_class'] == ref_class) | (df['PHQ_class'] == cls)
    y_bin = (df.loc[mask, 'PHQ_class'] == cls).astype(int).values
    Xb = X_design[mask.values]

    beta, cov = firth_logit(Xb, y_bin)
    se = np.sqrt(np.diag(cov))
    z = np.divide(beta, se, out=np.zeros_like(beta), where=se > 0)
    pvals = 2 * (1 - norm.cdf(np.abs(z)))

    convergence_log.append((cls, mask.sum(), int(y_bin.sum())))

    for i, col in enumerate(indicator_col_names):
        if col == 'const':
            continue
        b, s = beta[i], se[i]
        indicator_stats.setdefault(col, {})[cls] = (
            np.exp(b), np.exp(b - 1.96 * s), np.exp(b + 1.96 * s), pvals[i]
        )

print("Fitted Firth bias-reduced logistic regression, one-vs-reference. Classes vs reference:", list(eq_classes))
for cls, n, n_pos in convergence_log:
    print(f"  {cls}: n={n}, n_positive={n_pos}")


In [ ]:
# Human-readable variable labels (edit freely to match your thesis wording)
var_labels = {
    'age_class':                   'Age group',
    'B6_relationship_with_family': 'Relationship with family',
    'weight_class':                'Weight class',
    'D3_device':                   'Primary device',
    'D5_parents_screen_time':      "Parents' screen time",
    'F1_academic_satisfaction':    'Academic satisfaction',
    'G1_mood_swings':              'Mood swings',
    'G2_anxious_without_device':   'Anxious without device',
    'G4_isolation':                'Feelings of isolation',
    'G5_negative_mental':          'Negative mental state',
    'G6_panic_attack':             'Panic attacks',
    'H1_mobile_while_eating':      'Mobile use while eating',
}

# Build the long-form publication table:
# one row per (variable, level), columns grouped per outcome class
records = []
for feat in features:
    ref = ref_levels[feat]
    label = var_labels.get(feat, feat)
    records.append({
        'Variables': f"{label} (ref: {ref})",
        'is_header': True,
    })
    for indicator_col, level in var_group_map[feat]:
        row = {'Variables': level, 'is_header': False}
        for cls in eq_classes:
            OR, lo, hi, p = indicator_stats[indicator_col][cls]
            row[f"{cls}__P"]  = p
            row[f"{cls}__OR"] = OR
            row[f"{cls}__lo"] = lo
            row[f"{cls}__hi"] = hi
        records.append(row)

pub_table = pd.DataFrame(records)
print(f"Publication table shape: {pub_table.shape}")
pub_table.head(10)


In [18]:
# Render publication-style PDF: merged header groups (one per outcome class),
# sub-columns P-value / OR / 95% CI Lower / Upper — matching journal table layout
import os
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

os.makedirs("report_v3", exist_ok=True)
output_pdf_pub = "report_v3/mlr_publication_table_phq9.pdf"

ref_class = class_names[0]
n_groups = len(eq_classes)

# Column layout: Variables | (P-value, OR, Lower, Upper) per class
sub_cols_per_class = ['P-value', 'OR', 'Lower', 'Upper']
var_col_width = 0.22
group_col_width = (1 - var_col_width) / n_groups
sub_col_width = group_col_width / len(sub_cols_per_class)
col_widths = [var_col_width] + [sub_col_width] * (len(sub_cols_per_class) * n_groups)

def fmt_p(p):
    return f"{p:.2f}" if p >= 0.005 else "<.01"

def build_page_rows(page_df):
    header2 = ["Variables"] + sub_cols_per_class * n_groups
    body = []
    for _, r in page_df.iterrows():
        if r['is_header']:
            row = [r['Variables']] + [""] * (len(sub_cols_per_class) * n_groups)
        else:
            row = ["    " + r['Variables']]
            for cls in eq_classes:
                row += [fmt_p(r[f"{cls}__P"]), f"{r[f'{cls}__OR']:.3f}",
                        f"{r[f'{cls}__lo']:.3f}", f"{r[f'{cls}__hi']:.3f}"]
        body.append(row)
    return header2, body

rows_per_page = 16
pages = [pub_table.iloc[i:i+rows_per_page] for i in range(0, len(pub_table), rows_per_page)]

with PdfPages(output_pdf_pub) as pdf:
    for page_num, page_df in enumerate(pages):
        n_rows = len(page_df)
        fig_h = 1.6 + n_rows * 0.32
        fig, ax = plt.subplots(figsize=(20, fig_h))
        ax.axis('off')

        header2, body = build_page_rows(page_df)
        table_data = [header2] + body

        tbl = ax.table(cellText=table_data, colWidths=col_widths, loc='center', cellLoc='center',
                        bbox=[0, 0, 1, 0.92])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7.5)

        n_cols = len(header2)

        # Style sub-header row (row 0): P-value/OR/Lower/Upper
        for c in range(n_cols):
            cell = tbl[0, c]
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold', fontsize=7)

        # Style body rows
        for r_idx, (_, r) in enumerate(page_df.iterrows(), start=1):
            for c in range(n_cols):
                cell = tbl[r_idx, c]
                cell.set_edgecolor('#cccccc')
                if r['is_header']:
                    cell.set_facecolor('#dbe5f1')
                    if c == 0:
                        cell.set_text_props(fontweight='bold', ha='left')
                else:
                    cell.set_facecolor('white')
                    if c == 0:
                        cell.set_text_props(ha='left')

        # Merged outcome-class group header row, placed just above the table (below the title)
        for g, cls in enumerate(eq_classes):
            x0 = var_col_width + g * group_col_width
            ax.text(x0 + group_col_width / 2, 0.955, f"{cls}  (vs. {ref_class})",
                    transform=ax.transAxes, ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold')

        total_pages = len(pages)
        ax.set_title(
            f"Multinomial Logistic Regression (Firth bias-reduced) — PHQ-9 Depression Severity\n"
            f"Outcome: PHQ_class  (Reference category: '{ref_class}')   |   Page {page_num+1} of {total_pages}",
            fontsize=10, fontweight='bold', pad=42, y=1.0
        )
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f"Saved: {output_pdf_pub}  ({len(pages)} page(s), {len(pub_table)} rows)")


Saved: report_v3/mlr_publication_table_phq9.pdf  (3 page(s), 43 rows)
